# Agentic RAG: Router_retriever System

This notebook builds an agentic retrieval-augmented generation system centered on a router agent and a retriever agent: the router agent classifies an incoming question to decide the best source of information, then the retriever agent answers the question by pulling context from a local PDF, performing a live web search, or falling back to a direct LLM call when neither retrieval path is needed.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "your-tavily-api-key")
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER", "onnx")

os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)
os.environ.setdefault("OPENAI_MODEL", OPENAI_MODEL)
os.environ.setdefault("TAVILY_API_KEY", TAVILY_API_KEY)
os.environ.setdefault("EMBEDDING_PROVIDER", EMBEDDING_PROVIDER)

if OPENAI_API_KEY == "your-openai-api-key":
    raise ValueError(
        "OPENAI_API_KEY is still set to its placeholder value. "
        "Fill in a real key in your .env file."
    )
if TAVILY_API_KEY == "your-tavily-api-key":
    raise ValueError(
        "TAVILY_API_KEY is still set to its placeholder value. "
        "Fill in a real key in your .env file."
    )

In [4]:
from crewai import LLM

llm = LLM(
    model=OPENAI_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    max_tokens=512,
)

In [5]:
from crewai_tools import PDFSearchTool


def _notebook_dir():
    vsc_path = globals().get("__vsc_ipynb_file__")
    if vsc_path:
        return os.path.dirname(os.path.abspath(vsc_path))
    return os.path.abspath(os.getcwd())


PDF_PATH = os.path.abspath(
    os.path.join(_notebook_dir(), "data", "pdfs", "trasformer_research_paper-dataset.pdf")
)

if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(
        f"Expected PDF not found at {PDF_PATH}. "
        "Place the source PDF in data/pdfs/ before running this cell."
    )

pdf_search_tool = PDFSearchTool(
    pdf=PDF_PATH,
    config=dict(
        embedder=dict(
            provider=EMBEDDING_PROVIDER,
        ),
        vectordb=dict(
            provider="chromadb",
            config=dict(
                batch_size=1,
            ),
        ),
    ),
)

In [6]:
import requests
from crewai.tools import BaseTool


class TavilySearchTool(BaseTool):
    name: str = "Web_Search"
    description: str = (
        "Searches the live web via Tavily for up-to-date information. "
        "Input should be a natural-language search query."
    )

    def _run(self, query: str) -> str:
        try:
            response = requests.post(
                "https://api.tavily.com/search",
                json={
                    "api_key": TAVILY_API_KEY,
                    "query": query,
                    "max_results": 3,
                },
                timeout=15,
            )
        except requests.exceptions.RequestException as exc:
            return f"Web search request failed: {exc}"

        if not response.ok:
            try:
                error_detail = response.json().get("error", response.text)
            except ValueError:
                error_detail = response.text
            return f"Web search unavailable ({response.status_code}): {error_detail}"

        results = response.json().get("results", [])
        if not results:
            return "Web search returned no results."

        return "\n".join(
            f"{result.get('title', 'Untitled')} - {result.get('url', '')}"
            for result in results
        )


web_search_tool = TavilySearchTool()

In [7]:
from datetime import datetime

TRACE_LOG = []


def log_event(step, details):
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "step": step,
        "details": details,
    }
    TRACE_LOG.append(entry)
    print(f"[{entry['timestamp']}] {step}: {details}")
    return entry


PDF_KEYWORDS = (
    "pdf", "document", "paper", "report", "section", "chapter",
    "according to", "in the document",
)
WEB_KEYWORDS = (
    "latest", "current", "recent", "today", "now", "news", "update", "online",
)


def route_question(question):
    lowered = question.lower()
    if any(keyword in lowered for keyword in PDF_KEYWORDS):
        route = "pdf"
    elif any(keyword in lowered for keyword in WEB_KEYWORDS):
        route = "web"
    else:
        route = "direct"
    log_event("route_question", {"question": question, "route": route})
    return route


def _trim_response(text, limit=1800):
    if text is None:
        return text
    if len(text) <= limit:
        return text
    return text[:limit] + "[truncated]"


def _is_web_error(result):
    if not isinstance(result, str):
        return False
    return result.startswith("Web search unavailable") or result.startswith(
        "Web search request failed"
    )

In [8]:
def _call_llm(prompt):
    return llm.call(prompt)


def retrieve_answer(route, question):
    log_event("dispatch", {"route": route, "question": question})

    if route == "pdf":
        raw_result = pdf_search_tool.run(query=question)
        trimmed_result = _trim_response(str(raw_result))
        log_event("pdf_retrieval", {"result": trimmed_result})

        refine_prompt = (
            "You are answering a question using only the PDF excerpts below. "
            "Write one short paragraph, then up to 3 bullet points of supporting "
            "evidence quoted or paraphrased from the excerpts.\n\n"
            f"Question: {question}\n\n"
            f"PDF excerpts:\n{trimmed_result}"
        )
        try:
            result = _call_llm(refine_prompt)
            log_event(
                "pdf_refinement", {"answer_preview": _trim_response(str(result), 200)}
            )
        except Exception as exc:
            log_event("pdf_refinement_failed", {"error": str(exc)})
            result = trimmed_result

    elif route == "web":
        raw_result = web_search_tool.run(query=question)
        if _is_web_error(raw_result):
            log_event("web_search_failed", {"result": raw_result})
            fallback_prompt = (
                "Real-time web search was unavailable, so current information "
                "could not be verified. Answer the question from your own "
                "knowledge, and explicitly tell the user that real-time "
                "verification was not available and the answer may be out of "
                "date.\n\n"
                f"Question: {question}"
            )
            result = _call_llm(fallback_prompt)
            log_event(
                "web_fallback_direct",
                {"answer_preview": _trim_response(str(result), 200)},
            )
        else:
            log_event("web_search", {"result": _trim_response(str(raw_result))})
            result = raw_result

    else:
        result = _call_llm(question)
        log_event("direct_llm", {"answer_preview": _trim_response(str(result), 200)})

    log_event("tool_completed", {"route": route, "char_count": len(str(result))})
    return result

In [9]:
test_questions = [
    "According to the paper, what does the document say about the transformer architecture?",
    "What is the latest AI news today?",
    "What is the capital of France?",
]

for question in test_questions:
    route = route_question(question)
    answer = retrieve_answer(route, question)
    print(f"\nQ: {question}\nRoute: {route}\nA: {answer}\n{'-' * 60}")

print("\n=== TRACE LOG ===")
for entry in TRACE_LOG:
    print(entry)

[2026-09-09T13:30:48] route_question: {'question': 'According to the paper, what does the document say about the transformer architecture?', 'route': 'pdf'}
[2026-09-09T13:30:48] dispatch: {'route': 'pdf', 'question': 'According to the paper, what does the document say about the transformer architecture?'}


[2026-09-09T13:30:48] pdf_retrieval: {'result': 'Relevant Content:\n\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\n\ndescribed in section 3.2.\n\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\n\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\n\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\n\ntextual entailment and learning task-independent sentence representations [4, 27, 28, 22].\n\nEnd-to-end memory networks are based on a recurrent attention mechanism instead of sequence-\n\naligned recurrence and have been shown to perform well on simple-language question answering and\n\nlanguage modeling tasks [34].\n\nTo the best of our knowledge, however, the Transformer is the first transduction model relying\n\nentirely on self-attention to compute representations of its input an

[2026-09-09T13:30:50] pdf_refinement: {'answer_preview': 'The document describes the transformer architecture as a groundbreaking model that utilizes self-attention mechanisms to process input and output sequences without relying on traditional recurrent neu[truncated]'}
[2026-09-09T13:30:50] tool_completed: {'route': 'pdf', 'char_count': 1026}

Q: According to the paper, what does the document say about the transformer architecture?
Route: pdf
A: The document describes the transformer architecture as a groundbreaking model that utilizes self-attention mechanisms to process input and output sequences without relying on traditional recurrent neural networks (RNNs) or convolutional layers. This innovative approach allows the transformer to compute representations of sequences more effectively, making it suitable for various tasks such as reading comprehension and summarization.

- The transformer is noted as the first transduction model that relies entirely on self-attention for computin

[2026-09-09T13:30:53] web_search: {'result': 'Latest AI News January 2026: Samsung AI Smartphones & Top Trends - https://aitoolmind.com/latest-ai-news-today-january-2026\nlatest ai news today: Latest News & Videos, Photos about latest ai news today | The Economic Times - Page 1 - https://economictimes.indiatimes.com/topic/latest-ai-news-today\nLatest ai news today - Latest latest ai news today , Information & Updates - Marketing & Advertising -ET BrandEquity - https://brandequity.economictimes.indiatimes.com/tag/latest+ai+news+today'}
[2026-09-09T13:30:53] tool_completed: {'route': 'web', 'char_count': 490}

Q: What is the latest AI news today?
Route: web
A: Latest AI News January 2026: Samsung AI Smartphones & Top Trends - https://aitoolmind.com/latest-ai-news-today-january-2026
latest ai news today: Latest News & Videos, Photos about latest ai news today | The Economic Times - Page 1 - https://economictimes.indiatimes.com/topic/latest-ai-news-today
Latest ai news today - Latest lates

[2026-09-09T13:30:55] direct_llm: {'answer_preview': 'The capital of France is Paris.'}
[2026-09-09T13:30:55] tool_completed: {'route': 'direct', 'char_count': 31}

Q: What is the capital of France?
Route: direct
A: The capital of France is Paris.
------------------------------------------------------------

=== TRACE LOG ===
{'timestamp': '2026-09-09T13:30:48', 'step': 'route_question', 'details': {'question': 'According to the paper, what does the document say about the transformer architecture?', 'route': 'pdf'}}
{'timestamp': '2026-09-09T13:30:48', 'step': 'dispatch', 'details': {'route': 'pdf', 'question': 'According to the paper, what does the document say about the transformer architecture?'}}
{'timestamp': '2026-09-09T13:30:48', 'step': 'pdf_retrieval', 'details': {'result': 'Relevant Content:\n\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\n\ndescribed in section 3.2.\n\nSelf-attention, sometimes called intra-attention is an a